In [1]:
import torch
import torchvision
import torch.nn as nn
import torchvision.transforms as transforms
import numpy as np
import random
import torch.optim as optim

### Seed + device

In [2]:
seed = 200

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
elif torch.backends.mps.is_available():
    torch.mps.manual_seed(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

### Sprawdzenie średniej i odchylenia dla każdego kanału

In [3]:
transform = transforms.Compose([transforms.ToTensor()])
dataset = torchvision.datasets.ImageFolder("train/", transform=transform)

map_class_to_idx = dataset.class_to_idx

all_samples = torch.stack([sample[0] for sample in dataset])
print(f"Data norm per channel: mean={all_samples.mean(axis=(0,2,3))} | std={all_samples.std(axis=(0,2,3))}")

Data norm per channel: mean=tensor([0.5204, 0.4950, 0.4381]) | std=tensor([0.2841, 0.2770, 0.2974])


### Transformacje na zbiorach

In [ ]:
train_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomCrop(64, 4),
        transforms.RandomRotation(24),
        transforms.ToTensor(),
        transforms.RandomErasing(0.4),
        transforms.Normalize((0.5204, 0.4950, 0.4381), (0.2841, 0.2770, 0.2974))])

# TODO: sprawdzic, czy wszystkie transformacje sa potrzebne i daja dobre wyniki, ewentualnie dac nowe

val_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5204, 0.4950, 0.4381), (0.2841, 0.2770, 0.2974))])

train_dataset = torchvision.datasets.ImageFolder("train/", transform=train_transform)
val_dataset = torchvision.datasets.ImageFolder("train/", transform=val_transform)

dataset_size = len(train_dataset)
train_size = int(0.85 * dataset_size)
val_size = dataset_size - train_size

indices = torch.randperm(dataset_size)

train_subset = torch.utils.data.Subset(train_dataset, indices[:train_size])
val_subset = torch.utils.data.Subset(val_dataset, indices[train_size:])

print(f"Train: {len(train_subset)}")
print(f"Val: {len(val_subset)}")


Train: 74809
Val: 13202


### Dataloadery do treningu i walidacji

In [5]:
train_loader = torch.utils.data.DataLoader(train_subset, batch_size=128, shuffle=True, num_workers=4)
val_loader = torch.utils.data.DataLoader(val_subset, batch_size=128, shuffle=False, num_workers=4)

In [ ]:
class ConvLayer(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.act = nn.GELU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.act(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.act(x)
        x = self.pool(x)
        return x


class ConvNet(nn.Module):
    def __init__(self, n_classes):
        super().__init__()

        self.conv_layer1 = ConvLayer(3, 32)
        self.conv_layer2 = ConvLayer(32, 64)
        self.conv_layer3 = ConvLayer(64, 128)
        self.conv_layer4 = ConvLayer(128, 256)
        
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.dense_layer = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256, n_classes)
        )

        # TODO: zmienic parametry AdaptiveAvgPool np. na 2x2 i dodac warstwe liniowa do dense_layer
    
    def forward(self, x):
        x = self.conv_layer1(x)
        x = self.conv_layer2(x)
        x = self.conv_layer3(x)
        x = self.conv_layer4(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.dense_layer(x)
        return x

In [7]:
def calc_accuracy(predictions: np.ndarray, targets: np.ndarray, n_classes=50):
    assert len(predictions) == len(targets)
    accuracies = []
    for i in range(n_classes):
        accuracies.append((predictions[targets == i] == i).sum() / (targets == i).sum())
    return np.mean(accuracies)

In [8]:
class EarlyStopping:
    def __init__(self, patience=5, delta=0):
        self.patience = patience
        self.delta = delta
        self.best_score = None
        self.early_stop = False
        self.counter = 0
        self.best_model_state = None

    def __call__(self, val_loss, model):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.best_model_state = model.state_dict()
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.best_model_state = model.state_dict()
            self.counter = 0

    def load_best_model(self, model):
        model.load_state_dict(self.best_model_state)

In [9]:
def train(model, train_loader, val_loader, criterion, optimizer, scheduler, early_stopping, epochs):
    for epoch in range(epochs):
        print(f"Epoch {epoch+1}/{epochs}")

        model.train()
        train_loss = 0.0
        train_predictions = []
        train_targets = []

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            train_predictions.append(predicted.cpu())
            train_targets.append(labels.cpu())
        
        train_acc = calc_accuracy(torch.cat(train_predictions).numpy(), torch.cat(train_targets).numpy())
        train_loss = train_loss / len(train_loader)

        model.eval()
        val_loss = 0.0
        val_predictions = []
        val_targets = []

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)
            
                val_loss += loss.item()

                _, predicted = torch.max(outputs.data, 1)
                val_predictions.append(predicted.cpu())
                val_targets.append(labels.cpu())

        val_acc = calc_accuracy(torch.cat(val_predictions).numpy(), torch.cat(val_targets).numpy())
        val_loss = val_loss / len(val_loader)

        early_stopping(val_loss, model)
        if early_stopping.early_stop:
            print("Early stopping")
            break

        scheduler.step()

        print(f"Train loss: {train_loss:.3f}, accuracy: {train_acc*100:.2f}%")
        print(f"Val loss: {val_loss:.3f}, accuracy: {val_acc*100:.2f}%\n")
    
    early_stopping.load_best_model(model)


In [ ]:
EPOCHS = 100
model = ConvNet(len(map_class_to_idx)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
early_stopping = EarlyStopping(patience=5, delta=0)
# TODO: sprawdzic, czy early stopping z wiekszym patience da cos ciekawego w np. 100 epokach

In [11]:
train(model, train_loader, val_loader, criterion, optimizer, scheduler, early_stopping, EPOCHS)

Epoch 1/100
Train loss: 3.121, accuracy: 16.54%
Val loss: 2.685, accuracy: 25.53%

Epoch 2/100
Train loss: 2.701, accuracy: 26.26%
Val loss: 2.536, accuracy: 29.80%

Epoch 3/100
Train loss: 2.438, accuracy: 32.99%
Val loss: 2.219, accuracy: 38.18%

Epoch 4/100
Train loss: 2.246, accuracy: 37.94%
Val loss: 1.985, accuracy: 43.92%

Epoch 5/100
Train loss: 2.097, accuracy: 41.60%
Val loss: 1.921, accuracy: 46.30%

Epoch 6/100
Train loss: 1.980, accuracy: 44.90%
Val loss: 1.708, accuracy: 51.56%

Epoch 7/100
Train loss: 1.881, accuracy: 47.63%
Val loss: 1.640, accuracy: 53.01%

Epoch 8/100
Train loss: 1.796, accuracy: 49.79%
Val loss: 1.640, accuracy: 53.57%

Epoch 9/100
Train loss: 1.724, accuracy: 51.79%
Val loss: 1.482, accuracy: 57.76%

Epoch 10/100
Train loss: 1.658, accuracy: 53.53%
Val loss: 1.463, accuracy: 58.38%

Epoch 11/100
Train loss: 1.599, accuracy: 55.14%
Val loss: 1.416, accuracy: 60.06%

Epoch 12/100
Train loss: 1.552, accuracy: 56.44%
Val loss: 1.455, accuracy: 59.30%

E

In [ ]:
# TODO: (opcjonalne) dodac wykresy loss i accuracy na train i val

In [ ]:
# TODO: przepuscic test/ przez model i zapisac do csv zgodnie z trescia